# 运行命令速查 · bash_orders

> 配套项目：`cuda_basic/vector_add`
> 整理时间：2026-10-07
>
> 用途：把这一路用到过、踩过坑的**所有命令行**汇总成一册。
> 忘了怎么重新编译、怎么批量跑实验、怎么推 GitHub —— 先翻这里。

## 目录

| 节 | 内容 |
|---|---|
| §0 | 路径约定（后面所有命令都基于它） |
| §1 | 编译与重新编译 |
| §2 | 运行程序：参数与三个陷阱 |
| §3 | 一条命令跑多次实验 + 存文件 |
| §4 | 把结果提炼成 CSV |
| §5 | 汇总：每个配置的最优成绩 |
| §6 | `bench.sh` 完整脚本 |
| §7 | 在本 Notebook 里直接跑扫描（可执行） |
| §8 | Git 推送到 GitHub（SSH 方案） |
| §9 | 诊断命令（网络 / ignore / 性能） |
| §10 | 坑速查表 |

## §0 路径约定

⚠️ 本 Notebook 在 `cuda_basic/vector_add/bash_orders/`，而可执行文件在 `cuda_basic/build/bin/`。
**路径差了两层**，所以每条命令前面都先设好变量，避免相对路径写错。

```bash
CUDABASIC=~/projects/ai-infra/cuda_basic
VECTORADD=$CUDABASIC/vector_add
BIN=$CUDABASIC/build/bin/my_vector_add
LOGDIR=$VECTORADD/logs          # 所有实验产物统一放这里
N=$((1<<26))                    # 默认规模 = 67108864
```

| 变量 | 含义 |
|---|---|
| `CUDABASIC` | CMake 工程根目录（`CMakeLists.txt` 所在的那一层） |
| `VECTORADD` | 本次练习的目录（源码 + 笔记 + 日志都在这里面） |
| `BIN` | 可执行文件。受 `CMAKE_RUNTIME_OUTPUT_DIRECTORY` 控制，统一落在 `build/bin/` |
| `LOGDIR` | **实验产物目录**：ncu 报告、批量测试的原始输出、CSV 都放这里 |
| `N` | 元素个数，默认 $2^{26}$ = 67,108,864 |

---

### 📁 日志到底该放哪？（一个容易搞错的结构问题）

**产物必须跟着「它所属的练习」走，不能放到工程根目录。**

```
cuda_basic/                 ← CMake 工程根
├── CMakeLists.txt
├── build/                  ← 编译产物（.gitignore 忽略）
├── hello_cuda/             ← 练习 1
│   └── logs/               ← 练习 1 的产物
└── vector_add/             ← 练习 2
    ├── vector_add.cu
    ├── my_vector_add.cu
    ├── my_vector_add_notes.md
    ├── bash_orders/        ← 命令速查（本 Notebook）
    └── logs/               ← ✅ 练习 2 的产物
        └── ncu/
```

❌ **错**：`cuda_basic/logs/ncu/`
—— 它和 `vector_add/` **同级**，装的却是 vector_add 的报告。

两个后果：

1. **以后会糊成一团** —— 再加 `reduction`、`sgemm`，它们的报告会全挤进同一个目录，分不清谁是谁
2. **删不干净** —— `rm -rf vector_add/` 之后，它的日志还孤零零留在外面

✅ **对**：每个练习目录**自包含** —— 源码、笔记、日志都在自己家里。

> ⚠️ 这和「`build/bin` 集中放二进制」不矛盾：
> 二进制是**构建系统**的产物（CMake 说了算），实验数据是**你**的产物（你自己组织）。

---

### 建目录 + 确认二进制存在

```bash
mkdir -p "$LOGDIR/ncu"
ls -l "$CUDABASIC/build/bin/"
```

预期看到 `my_vector_add` 和 `vector_add` 两个可执行文件。


In [ ]:
# §0 环境自检：确认二进制存在，并跑一次冒烟测试
import pathlib, subprocess

HOME = pathlib.Path.home()
CUDABASIC = HOME / "projects/ai-infra/cuda_basic"
BIN = CUDABASIC / "build/bin/my_vector_add"

print("CUDABASIC :", CUDABASIC)
print("BIN       :", BIN)
print("存在吗    :", BIN.exists())
print()

if BIN.exists():
    r = subprocess.run(
        [str(BIN), str(1 << 20), "512"],
        capture_output=True, text=True, cwd=CUDABASIC,
    )
    print("exit code :", r.returncode)
    print("---- stdout ----")
    print(r.stdout)
    if r.stderr:
        print("---- stderr ----")
        print(r.stderr)
else:
    print("先编译：cmake -S . -B build && cmake --build build --target my_vector_add")

## §1 编译与重新编译

| 你改了什么 | 要跑什么 |
|---|---|
| 只改了 `.cu` 里的代码 | `cmake --build build --target my_vector_add` |
| 新增 / 删除 / 改名 `.cu`，或改了 `CMakeLists.txt` | 先 `cmake -S . -B build`，再 build |
| 想彻底从头再来 | `rm -rf build` → 配置 → build |

```bash
cd $CUDABASIC

# 1) 只改了 .cu（最常用）
cmake --build build --target my_vector_add

# 2) 编译 + 立刻运行，一步到位
cmake --build build --target my_vector_add && $BIN $N 1024

# 3) 改了 CMakeLists.txt / 加了新文件
cmake -S . -B build && cmake --build build

# 4) 从头再来
rm -rf build && cmake -S . -B build && cmake --build build
```

> `--target my_vector_add` 表示**只编这一个**，不去碰 `hello_cuda` 和 `vector_add`，快很多。

### 怎么判断「真的重新编译了」

编译成功时的输出：

```
[ 50%] Building CUDA object .../my_vector_add.cu.o     ← 关键：有这一行 = 真的在编
[100%] Built target my_vector_add
```

如果**只**看到 `Built target` 而没有 `Building` → **它认为文件没变，压根没编**。

> 最常见的原因只有一个：**改完代码忘了 `Ctrl+S` 保存**。
> 看到「改了没生效」，第一件事不是怀疑 CMake，是先保存。

## §2 运行程序：参数与三个陷阱

```bash
# 用法
$BIN [N] [thread_per_block]

# 冒烟测试（小规模，秒出）
$BIN $((1<<20)) 512

# 标准性能测试
$BIN $((1<<26)) 256      # 预期 ~3.53 ms / ~228 GB/s
$BIN $((1<<26)) 1024     # 预期 ~3.28 ms / ~245.8 GB/s

# 边界测试：N 故意不是 block 的整数倍
$BIN $((1<<26)) 512
```

### ⚠️ 陷阱 1：shell 不会算数

| 写法 | 实际发生的事 |
|---|---|
| `$BIN 1<<25 2^10` | `<<` 被 bash 当成 **heredoc 重定向** → 卡在 `>` 续行提示符等你输入 |
| `$BIN '1<<25' '2^10'` | 字符串传进去了，但 `atoi` 只解析开头数字 → **N=1, block=2** |
| `$BIN $((1<<25)) $((2**10))` | ✅ 正确：N=33554432, block=1024 |

### ⚠️ 陷阱 2：`^` 不是乘方，是异或

```bash
echo $((1<<25))   # 33554432   ✅ 移位没问题
echo $((2^10))    # 8          ❌ 0010 XOR 1010 = 1000 = 8
echo $((2**10))   # 1024       ✅ 乘方要用 **
```

### ⚠️ 陷阱 3：`atoi` 不会报错

```c
atoi("1<<25")   → 1        // 读到 '<' 就停
atoi("2^10")    → 2        // 读到 '^' 就停
atoi("1024abc") → 1024
atoi("abc")     → 0
```

**所以程序开头那行 `printf("N = ... block = ...")` 必须保留** ——
它是唯一能让你发现「参数被吃掉了」的手段。

## §3 一条命令跑多次实验 + 存文件

### 先认识 4 个符号

| 符号 | 作用 |
|---|---|
| `>` | **覆盖**写入（每次清空） |
| `>>` | **追加**写入 |
| `2>&1` | 把 **stderr** 也收进文件 |
| `tee` | 屏幕 + 文件同时写（`tee -a` 追加） |

> ⚠️ **头号坑**：`for ...; do ./prog > out.txt; done`
> 每一轮都用 `>` **覆盖**，结果**只剩最后一次**。

> ⚠️ 你的 `CUDA_CHECK` 是用 `fprintf(stderr, ...)` 报错的。
> **不加 `2>&1`，错误信息只会打在屏幕上，不会进文件。**

---

### 第 1 级：一个循环，一个文件

```bash
cd $CUDABASIC
: > "$LOGDIR/results.txt"                        # 冒号是空命令，这行 = 清空文件
for b in 128 256 512 1024; do
    $BIN $N $b >> "$LOGDIR/results.txt"
done
```

- 用 `: > file` 而不是 `> file`：**它一定不会 hang 住等输入**
- 缺点：输出很长，4 个配置混在一起，不好对比

---

### 第 2 级：每个配置一个文件，屏幕也能看到

```bash
cd $CUDABASIC
mkdir -p "$LOGDIR/raw"
for b in 128 256 512 1024; do
    $BIN $N $b 2>&1 | tee "$LOGDIR/raw/block_${b}.txt"
done
```

`tee` 让你边跑边看到进度，同时留下证据。想安静跑就把 `| tee ...` 换成 `> ... 2>&1`。

---

### 第 3 级：重复多次 —— 注意「交替跑」

```bash
cd $CUDABASIC
mkdir -p "$LOGDIR/raw"
for rep in 1 2 3 4 5; do
  for b in 128 256 512 1024; do
    $BIN $N $b 2>&1 | tee -a "$LOGDIR/raw/block_${b}.txt"
  done
done
```

**嵌套顺序很重要：`rep` 在外、配置在内。**

| 写法 | 效果 |
|---|---|
| `for b { for rep }` | 配置 A 连跑 5 次 → 再跑 B。**如果 GPU 越来越热，A 吃亏、B 占便宜** |
| **`for rep { for b }`** | A B C D A B C D… **交替跑**，热漂移平摊到所有配置，比较才公平 |

> 依据：同一份代码同一组参数，第一次跑 4.038 ms、第二次 3.529 ms —— **13% 的差距纯粹来自温度和时钟波动**。

---

### 第 4 级：只要你想要「一条命令」的极简版

```bash
cd $CUDABASIC && mkdir -p "$LOGDIR/raw" && for r in 1 2 3 4 5; do for b in 128 256 512 1024; do $BIN $N $b > "$LOGDIR/raw/b${b}_r${r}.txt" 2>&1; done; done; echo "完成，结果在 $LOGDIR"
```


## §4 把结果提炼成 CSV

原始输出太啰嗦，真正要对齐比较的只有几个数：

```bash
cd $CUDABASIC
mkdir -p "$LOGDIR/raw"
CSV="$LOGDIR/results.csv"
echo "block,rep,kernel_ms,bw_kernel,htd_ms,dth_ms,verify" > "$CSV"

for rep in 1 2 3 4 5; do
  for b in 128 256 512 1024; do
    raw="$LOGDIR/raw/b${b}_r${rep}.txt"
    $BIN $N $b > "$raw" 2>&1
    awk -v b="$b" -v r="$rep" '
      /kernel best/               { k = $(NF-1) }
      /kernel bandwith/           { w = $(NF-1) }
      /HostToDevice/ && /ms$/     { h = $(NF-1) }
      /DeviceToHost/ && /ms$/     { d = $(NF-1) }
      /^verify/                   { v = $0; sub(/^verify: */, "", v) }
      END { printf "%s,%s,%.4f,%.2f,%.4f,%.4f,\"%s\"\n", b, r, k, w, h, d, v }
    ' "$raw" >> "$CSV"
  done
done

column -t -s, "$CSV"        # 对齐打印
```

### `$(NF-1)` 是什么意思？

以 `kernel best       : 3.5280 ms` 为例，按空格切分后：

```
$1=kernel   $2=best   $3=:   $4=3.5280   $5=ms
NF = 5   →   $(NF-1) = $4 = 3.5280   ✅
```

其他几行同理（`kernel bandwith: 228.30 GB/s` → NF=3 → `$2`）。

### ⚠️ 为什么必须同时用「行首前缀」和「行尾单位」两个条件

程序现在的输出里，**有两行都以 `HostToDevice` 开头**：

```
HostToDevice copy : 1.6647 ms              ← 时间
HostToDevice copy bandweight: 4.79 GB/s    ← 带宽
```

如果只写 `/^HostToDevice/ { h = $(NF-1) }`，awk 会取**最后匹配到的那一行**，
也就是把带宽 `4.79` 当成时间写进 CSV —— **不报错，但数据全错**。

所以加上 `/ms$/` 和 `/GB\/s/` 两个维度来区分：

| 想抓什么 | 匹配条件 |
|---|---|
| H2D 时间 | `/HostToDevice/ && /ms$/` |
| H2D 带宽 | `/HostToDevice/ && /GB\/s/` |
| D2H 时间 | `/DeviceToHost/ && /ms$/` |
| D2H 带宽 | `/DeviceToHost/ && /GB\/s/` |

> 📌 靠**行首关键词**定位比靠列号稳，但**前提是关键词足够唯一**。
> 一旦程序输出里出现前缀相同、含义不同的行，就必须再加一个维度（这里是单位）。
> **这又是一次「不报错但数据错」的教训 —— 和那 3 个沉默 bug 同一类。**

### 一个可选的增强：加时间戳

发现某几轮带宽普遍偏低时，一看时间就能判断是不是机器变热了：

```bash
awk -v ts="$(date +%H:%M:%S)" -v b="$b" -v r="$rep" '...'
```


## §5 汇总：每个配置的最优成绩

```bash
cd $CUDABASIC
CSV="$LOGDIR/results.csv"

{ echo "block,runs,min_time_ms,max_bw_GBps"
  awk -F, 'NR>1 {
      if (!($1 in t) || $3+0 < t[$1]) t[$1] = $3+0
      if (!($1 in w) || $4+0 > w[$1]) w[$1] = $4+0
      n[$1]++
    }
    END { for (k in t) printf "%s,%d,%.3f,%.2f\n", k, n[k], t[k], w[k] }' "$CSV" \
    | sort -n
} | column -t -s,
```

输出长这样：

```
block  runs  min_time_ms  max_bw_GBps
128    5     3.527        228.30
256    5     3.523        228.60
512    5     3.519        228.90
1024   5     3.275        245.90
```

一眼就能看出 **block=1024 明显更快**。


## §6 `bench.sh` 完整脚本

把 §3～§5 固化成一个可复用脚本（推荐，你会反复用）。

```bash
#!/usr/bin/env bash
set -euo pipefail
#
# 用法：在 cuda_basic/ 目录下运行   ./bench.sh
# 产物：vector_add/logs/bench/   ← 跟着练习走，不放到工程根目录（见 §0）

BIN=${BIN:-./build/bin/my_vector_add}
N=$((1 << 26))
BLOCKS=(128 256 512 1024)
REPS=5
OUTDIR=vector_add/logs/bench

mkdir -p "$OUTDIR/raw"
CSV="$OUTDIR/results.csv"
echo "timestamp,block,rep,kernel_ms,bw_kernel,htd_ms,dth_ms,verify" > "$CSV"

for ((r = 1; r <= REPS; r++)); do
  for b in "${BLOCKS[@]}"; do
    raw="$OUTDIR/raw/b${b}_r${r}.txt"
    ts=$(date +%H:%M:%S)
    "$BIN" "$N" "$b" >"$raw" 2>&1 || echo "block=$b rep=$r 运行失败，见 $raw" >&2
    awk -v ts="$ts" -v b="$b" -v r="$r" '
      /kernel best/               { k = $(NF-1) }
      /kernel bandwith/           { w = $(NF-1) }
      /HostToDevice/ && /ms$/     { h = $(NF-1) }
      /DeviceToHost/ && /ms$/     { d = $(NF-1) }
      /^verify/                   { v = $0; sub(/^verify: */, "", v) }
      END { printf "%s,%s,%s,%.4f,%.2f,%.4f,%.4f,\"%s\"\n", ts, b, r, k, w, h, d, v }
    ' "$raw" >>"$CSV"
    printf "block=%-5s rep=%d  完成\n" "$b" "$r" >&2
  done
done

echo
{ echo "block,runs,min_time_ms,max_bw_GBps,fail"
  awk -F, 'NR>1 {
      gsub(/"/, "", $8)
      if (!($2 in t) || $4+0 < t[$2]) t[$2] = $4+0
      if (!($2 in w) || $5+0 > w[$2]) w[$2] = $5+0
      n[$2]++
      if ($8 !~ /^PASS/) f[$2]++
    }
    END { for (k in t) printf "%s,%d,%.3f,%.2f,%d\n", k, n[k], t[k], w[k], f[k]+0 }' "$CSV" \
    | sort -n
} | column -t -s,

echo
echo "原始数据：$CSV"
```

运行：

```bash
cd $CUDABASIC
chmod +x bench.sh
./bench.sh
```

### `set -euo pipefail` 三个开关

| 开关 | 作用 |
|---|---|
| `-e` | 任何命令返回非 0 就**立刻停止**，不会带着错误继续跑 |
| `-u` | 用了未定义的变量就报错（防止打错变量名后静默变成空字符串） |
| `-o pipefail` | 管道里**任何一环**失败都算失败（默认只看最后一环） |

### 两列很关键的输出

| 列 | 意义 |
|---|---|
| `timestamp` | 出现带宽普遍偏低时，一眼看出是不是机器变热了 |
| `fail` | 5 次里有几次没打印 `PASS`。**这一列必须是 0**，否则后面的性能数字全不作数 |

> ⚠️ 脚本里的 `BIN` 和 `OUTDIR` 都用了 `${BIN:-默认值}` 的写法：
> 环境变量优先，没设就用默认。所以你既可以 `cd cuda_basic && ./bench.sh`，
> 也可以 `BIN=/别的路径/程序 ./bench.sh`。


In [ ]:
# §7 在本 Notebook 里直接跑扫描（可执行版）
#
# 说明：这里故意用较小的 N（2^24）以便几秒内跑完。
# 想做正式测试，把 N 改成 1<<26，REPS 改成 5。
#
# ⚠️ 本单元格依赖上一个代码单元格里定义的 BIN / CUDABASIC，请按顺序从上往下运行。

import pathlib, subprocess, csv

N = 1 << 24           # 1<<26 是正式规模，但耗时较长
BLOCKS = [128, 256, 512, 1024]
REPS = 3

# 产物放 vector_add/logs/bench/ —— 跟着练习走，不放到工程根目录（见 §0）
OUT = CUDABASIC / "vector_add" / "logs" / "bench"
(OUT / "raw").mkdir(parents=True, exist_ok=True)

assert BIN.exists(), f"找不到可执行文件：{BIN}（先执行 §1 的编译命令）"

rows = []
for rep in range(1, REPS + 1):
    for b in BLOCKS:
        r = subprocess.run(
            [str(BIN), str(N), str(b)],
            capture_output=True, text=True, cwd=CUDABASIC,
        )
        (OUT / "raw" / f"b{b}_r{rep}.txt").write_text(r.stdout + r.stderr)

        rec = {"block": b, "rep": rep, "verify": "?",
               "kernel_ms": None, "bw_kernel": None,
               "htd_ms": None, "dth_ms": None}

        for line in r.stdout.splitlines():
            s = line.rstrip()
            if s.startswith("kernel best"):
                rec["kernel_ms"] = float(s.split()[-2])
            elif s.startswith("kernel bandwith"):
                rec["bw_kernel"] = float(s.split()[-2])
            # ⚠️ 注意要同时判断「行首前缀」和「行尾单位」：
            #    "HostToDevice copy : 1.66 ms"            → 时间
            #    "HostToDevice copy bandweight: 4.79 GB/s" → 带宽
            #    只按前缀匹配会取到后面那行，把带宽当成时间！
            elif s.startswith("HostToDevice") and s.endswith("ms"):
                rec["htd_ms"] = float(s.split()[-2])
            elif s.startswith("DeviceToHost") and s.endswith("ms"):
                rec["dth_ms"] = float(s.split()[-2])
            elif s.startswith("verify"):
                rec["verify"] = s.split(":", 1)[1].strip()

        rows.append(rec)

# 写 CSV
csv_path = OUT / "results.csv"
with csv_path.open("w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    w.writeheader()
    w.writerows(rows)

print(f"共 {len(rows)} 次实验，原始数据已写入 {csv_path}")
print()

# 汇总：每个 block 取最快的 kernel 时间 + 最高的带宽
summary = {}
for r in rows:
    b = r["block"]
    if b not in summary:
        summary[b] = {"runs": 0, "min_ms": float("inf"), "max_bw": 0.0, "fail": 0}
    s = summary[b]
    s["runs"] += 1
    if r["kernel_ms"] is not None:
        s["min_ms"] = min(s["min_ms"], r["kernel_ms"])
    if r["bw_kernel"] is not None:
        s["max_bw"] = max(s["max_bw"], r["bw_kernel"])
    if not r["verify"].startswith("PASS"):
        s["fail"] += 1

print(f"{'block':>6} {'runs':>5} {'min_ms':>10} {'max_bw':>10} {'fail':>5}")
print("-" * 42)
for b in sorted(summary):
    s = summary[b]
    print(f"{b:>6} {s['runs']:>5} {s['min_ms']:>10.3f} {s['max_bw']:>10.2f} {s['fail']:>5}")

print()
print("⚠️ 检查三件事：")
print("   1. fail 列必须全是 0")
print("   2. max_bw 不能超过 256.3 GB/s")
print(f"      （N={N} 时工作集 = 3 × {N*4/1024/1024:.0f} MiB，远超 L2，所以这条成立）")
print("   3. 小 N 时带宽可以超过 256.3 GB/s —— 那是数据驻留 L2 的正常现象，不是 bug")


## §8 Git 推送到 GitHub（SSH 方案）

### 为什么不能用 HTTPS

诊断结果：

| 测试 | 结果 | 结论 |
|---|---|---|
| `curl https://github.com` | 超时 | **HTTPS 走不通** |
| `curl https://www.baidu.com` | HTTP 200 | 网络本身正常 |
| `ssh -T git@github.com` | `Hi BillHerny!` | **SSH 通，且已认证为你的账号** ✅ |

→ 所以 remote 必须用 SSH 地址，不能用 `https://...`。

```bash
# 查看当前 remote
git remote -v

# 配成 SSH（如果还没配）
git remote add origin git@github.com:BillHerny/AI_Infra_learning.git
# 或改掉已有的
git remote set-url origin git@github.com:BillHerny/AI_Infra_learning.git
```

### 推送流程

```bash
cd ~/projects/ai-infra

# 1. 解锁 SSH 私钥（只做一次，之后本次会话不再问密码短语）
ssh-add ~/.ssh/id_ed25519

# 2. 对齐 GitHub 默认分支名
#    （本地原来叫 master，GitHub 默认叫 main）
git branch -M main

# 3. 推送
git push -u origin main
```

### 如果推送被拒

| 报错 | 原因 | 处理 |
|---|---|---|
| `rejected ... non-fast-forward` | 远端已有提交（建仓库时勾了 README） | 远端只有 README → `git push -u origin main --force`；想保留 → `git pull --rebase origin main` 再推 |
| `Permission denied (publickey)` | 私钥没解锁 | 重新 `ssh-add ~/.ssh/id_ed25519` |
| `Repository not found` | 仓库名拼错或没权限 | 核对 `BillHerny/AI_Infra_learning` |

### 提交前的两件事

```bash
# 看看这次会提交什么（重点确认没有 build/ 和 .venv/）
git add -A
git status --short

# 撤销最近一次提交，但保留改动
git reset --soft HEAD~1
```

### `.gitignore` 建议内容

```gitignore
build/
.venv/
.ipynb_checkpoints/
.DS_Store
*.swp
```

> ⚠️ 现在 `.venv/` 之所以没被提交，只是因为 `python -m venv` 在虚拟环境内部
> **自己写了一个含 `*` 的 `.gitignore`** —— 属于**碰巧**，不是你的配置在保护你。
> 哪天换种方式建环境，整个 venv（几百 MB）就会被提交进去。

## §9 诊断命令

### 网络

```bash
# 各站点连通性 + 耗时
for u in https://github.com https://www.baidu.com https://gitee.com; do
  curl -sS -o /dev/null -w "$u -> HTTP %{http_code}, %{time_total}s\n" --max-time 10 "$u"
done

# 检查有没有配代理
env | grep -i proxy
git config --global --get http.proxy

# 测试 GitHub 的 SSH（不加 BatchMode 才会问密码短语）
ssh -T git@github.com
ssh -T -p 443 git@ssh.github.com          # 22 端口被挡时试 443
```

### 安全探测（不卡在密码提示输入）

```bash
GIT_SSH_COMMAND='ssh -o BatchMode=yes' git ls-remote origin
GIT_TERMINAL_PROMPT=0 git ls-remote https://github.com/USER/REPO.git
```

### 确认哪些文件被忽略 / 会被提交

```bash
cat .gitignore

git status --short --ignored | grep '^!!'      # 列出被忽略的
git check-ignore -v cuda_basic/build           # 查是哪条规则忽略的
```

### 每次都要看输出的话

```bash
ls -l build/bin/                                # 二进制在不在
which nvcc && nvcc --version                    # CUDA 工具链
nvidia-smi --query-gpu=name,clocks.sm,clocks.mem,temperature.gpu --format=csv
```

### 性能分析

```bash
ncu --set full -k vectorAdd --launch-count 1 ./build/bin/my_vector_add
```

## §10 坑速查表

| # | 坑 | 症状 | 正确做法 |
|---|---|---|---|
| 1 | 循环里用 `>` | **只剩最后一次结果** | 用 `>>` 或每轮单独文件名 |
| 2 | 忘了 `2>&1` | 报错信息进不了文件 | `... > out.txt 2>&1` |
| 3 | `rep` 写在最内层 | 热漂移污染比较结果 | `rep` 在外、配置在内（交替跑） |
| 4 | 忘了保存就编译 | 只有 `Built target` 没有 `Building` | `Ctrl+S` |
| 5 | 改了 `CMakeLists.txt` 没重新配置 | target 不存在 / 新文件没编进去 | 先 `cmake -S . -B build` |
| 6 | `$BIN 1<<25 2^10` | 卡在 `>` 提示符，或在跑 N=1 | `$((1<<25)) $((2**10))` |
| 7 | 以为 `^` 是乘方 | `$((2^10))` = 8 | 用 `**` |
| 8 | 用 HTTPS 推 GitHub | 超时 / TLS 错误 | 改用 SSH 地址 |
| 9 | 一个 target 里两个 `main` | `multiple definition of 'main'` | 拆成两个 target |
| 10 | 带宽超过 256.3 GB/s | 假数据 | 检查 `best_time` 初值 |

---

### 一条物理规律，永远有效

> **超过硬件峰值的数字 = 代码 bug，不是机器牛。**
>
> 本机上限：带宽 **256.3 GB/s**、FP32 算力 **9.5 TFLOP/s**。
> 看到 `805.31 GB/s` 这种，第一反应应该是「我的测量代码坏了」。
>
> 附赠一条直觉：**看到「太圆」的数字（比如 `1.0000 ms`），先怀疑它是初值、默认值、没被更新的值。**